#Simple Gen Ai App using langchain

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPEN_AI_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")
os.environ["USER_AGENT"] = "my-langchain-app/1.0"

In [ ]:
## Data Ingestion - from website need to scrap data
from langchain_community.document_loaders import WebBaseLoader
url = "https://docs.langchain.com/langsmith/manage-prompts"
loader = WebBaseLoader(url)
documents = loader.load()

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/manage-prompts', 'title': 'Manage prompts - Docs by LangChain', 'language': 'en'}, page_content='Manage prompts - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationCreate and update promptsManage promptsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewQuickstartConceptsPolly AI assistantModel providersCreate and update promptsCreate a promptManage promptsManage prompts programmaticallyPrompt template formatConfigure prompt settingsUse tools in a promptInclude multimodal content in a promptWrite your prompt with AIConnect to modelsTutorialsOptimize a classifierSync prompts with GitHubTest multi-turn conversationsOn this pageCommit tagsCreate a tagMove a tagDelete a tagUse tags in codePrompt ow

In [2]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o")
print(llm)


d:\ai\langchain\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


profile={'name': 'GPT-4o', 'release_date': '2024-05-13', 'last_updated': '2024-08-06', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True} client=<openai.resources.chat.completions.completions.Completions object at 0x0000023538493C10> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002353898B150> root_client=<openai.OpenAI object at 0x00000235384936D0> root_async_client=<openai.AsyncOpenAI object at 0x000002353898AE90> model_name='gpt-4o' model_kwargs={} openai_api_key=SecretStr('**********') stream_usa

In [ ]:
# Load Data -> > Text Splitter chunks -> Embeddings -> VectorDB (Faiss) -> RetrievalQA
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = text_splitter.split_documents(documents)
docs



[Document(metadata={'source': 'https://docs.langchain.com/langsmith/manage-prompts', 'title': 'Manage prompts - Docs by LangChain', 'language': 'en'}, page_content='Manage prompts - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationCreate and update promptsManage promptsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewQuickstartConceptsPolly AI assistantModel providersCreate and update promptsCreate a promptManage promptsManage prompts programmaticallyPrompt template formatConfigure prompt settingsUse tools in a promptInclude multimodal content in a promptWrite your prompt with AIConnect to modelsTutorialsOptimize a classifierSync prompts with GitHubTest multi-turn conversationsOn this pageCommit tagsCreate a tagMove a tagDelete a tagUse tags in codePrompt ow

In [ ]:
##Convert this text into vector embeddings
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x000001E120EEC4D0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000001E123E69410>, model='text-embedding-ada-002', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [ ]:
from langchain_community.vectorstores import FAISS
db = FAISS.from_documents(docs, embeddings)


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
## QUERY
query = "Triggering a CI/CD pipeline in langsmith ?"
retriever = db.similarity_search(query)
retriever

In [ ]:
# Correct imports
# from langchain.chains.combine_documents import create_stuff_documents_chain
# from langchain.prompts import ChatPromptTemplate

# # Prompt
# prompt = ChatPromptTemplate.from_template(
#     """
#     Answer the following question based only on the provided context:

#     <context>
#     {context}
#     </context>

#     Question: {input}
#     """
# )

# # Document chain
# document_chain = create_stuff_documents_chain(
#     llm,
#     prompt
# )

ModuleNotFoundError: No module named 'langchain.chains'

In [ ]:
document_chain.invoke({
    "input": query, 
    "context": retriever[0].page_content
})

In [ ]:
from langchain_core.documents import Document
document_chain.invoke({
    "input": query, 
    "context": Document(page_content=retriever[0].page_content)
})
    